# 📱 Phone Detection & Monitoring System

A Computer Vision project that detects mobile phones, tracks them with ByteTrack, measures how long each phone remains visible, generates warnings for long detections, and creates a CSV monitoring log.

### Project pipeline

**Dataset → Annotation Cleaning → YOLOv8 Training → Detection → ByteTrack → Duration Monitoring → Alerts → CSV Logging → Final Video**

## 1. Install and Import Libraries

This notebook was developed in Google Colab.

In [ ]:
!pip install -q ultralytics lap

import os
import zipfile
import cv2
import random
import torch
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from google.colab import files
from ultralytics import YOLO

print("Libraries imported successfully!")

## 2. Upload and Extract the Dataset

Upload the YOLOv8-format dataset ZIP exported from Roboflow.

In [ ]:
uploaded = files.upload()

zip_path = next(
    (f"/content/{name}" for name in uploaded if name.lower().endswith(".zip")),
    None
)

if zip_path is None:
    raise FileNotFoundError("Please upload the dataset ZIP file.")

dataset_path = "/content/phone_dataset"

if os.path.exists(dataset_path):
    import shutil
    shutil.rmtree(dataset_path)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(dataset_path)

print("Dataset extracted successfully!")
print(os.listdir(dataset_path))

## 3. Inspect Dataset Structure

The dataset contains separate training, validation, and test splits.

In [ ]:
for root, dirs, files_list in os.walk(dataset_path):
    level = root.replace(dataset_path, "").count(os.sep)
    indent = "    " * level
    print(indent + os.path.basename(root) + "/")

## 4. Inspect Original Class Distribution

The original Roboflow dataset contained three class IDs:

- `0` → `6`
- `1` → `phone`
- `2` → `undefined`

The unusual annotations were manually inspected during dataset preparation and verified to represent phones.

In [ ]:
for split in ["train", "valid", "test"]:
    label_folder = os.path.join(dataset_path, split, "labels")
    counter = Counter()

    for file in os.listdir(label_folder):
        if file.endswith(".txt"):
            with open(os.path.join(label_folder, file), "r") as f:
                for line in f:
                    if line.strip():
                        counter[line.split()[0]] += 1

    print(f"{split.upper()}: {counter}")

## 5. Clean the Annotations

For this project, we use a **single class**:

`0 = phone`

The verified annotations are therefore converted to class `0`.

In [ ]:
for split in ["train", "valid", "test"]:
    label_folder = os.path.join(dataset_path, split, "labels")

    for file in os.listdir(label_folder):
        if not file.endswith(".txt"):
            continue

        path = os.path.join(label_folder, file)

        with open(path, "r") as f:
            lines = f.readlines()

        new_lines = []

        for line in lines:
            parts = line.strip().split()

            if len(parts) != 5:
                continue

            # All verified annotations represent phones.
            parts[0] = "0"
            new_lines.append(" ".join(parts) + "\n")

        with open(path, "w") as f:
            f.writelines(new_lines)

print("All annotations converted to class 0 = phone.")

## 6. Verify the Cleaned Dataset

In [ ]:
for split in ["train", "valid", "test"]:
    label_folder = os.path.join(dataset_path, split, "labels")
    counter = Counter()
    annotation_count = 0

    for file in os.listdir(label_folder):
        if file.endswith(".txt"):
            with open(os.path.join(label_folder, file), "r") as f:
                for line in f:
                    if line.strip():
                        counter[line.split()[0]] += 1
                        annotation_count += 1

    print(split.upper())
    print("Classes:", counter)
    print("Annotations:", annotation_count)
    print("-" * 40)

## 7. Create YOLO Dataset Configuration

In [ ]:
yaml_path = os.path.join(dataset_path, "data_phone.yaml")

yaml_content = f"""path: {dataset_path}

train: train/images
val: valid/images
test: test/images

nc: 1
names: ['phone']
"""

with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(yaml_content)

## 8. Visualize Training Annotations

Random samples are displayed with their bounding boxes to verify that phones are correctly annotated.

In [ ]:
split = "train"
image_folder = os.path.join(dataset_path, split, "images")
label_folder = os.path.join(dataset_path, split, "labels")

images = [
    f for f in os.listdir(image_folder)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

sample_images = random.sample(images, min(6, len(images)))

for image_name in sample_images:
    image_path = os.path.join(image_folder, image_name)
    label_path = os.path.join(
        label_folder,
        os.path.splitext(image_name)[0] + ".txt"
    )

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    h, w, _ = image.shape

    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()

                if len(parts) != 5:
                    continue

                _, x, y, bw, bh = map(float, parts)

                x_center = int(x * w)
                y_center = int(y * h)
                box_width = int(bw * w)
                box_height = int(bh * h)

                x1 = int(x_center - box_width / 2)
                y1 = int(y_center - box_height / 2)
                x2 = int(x_center + box_width / 2)
                y2 = int(y_center + box_height / 2)

                cv2.rectangle(
                    image, (x1, y1), (x2, y2), (255, 0, 0), 2
                )

                cv2.putText(
                    image, "phone",
                    (x1, max(y1 - 10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7, (255, 0, 0), 2
                )

    plt.figure(figsize=(8, 6))
    plt.imshow(image)
    plt.axis("off")
    plt.title(image_name)
    plt.show()

## 9. Check GPU Availability

The original training run used CPU because a GPU was not available in the Colab runtime.

In [ ]:
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Training will use CPU")

## 10. Train YOLOv8 Nano

Training configuration used in the project:

- Model: YOLOv8 Nano
- Epochs: 10
- Image size: 640
- Batch size: 8

In [ ]:
model = YOLO("yolov8n.pt")

results = model.train(
    data=yaml_path,
    epochs=10,
    imgsz=640,
    batch=8,
    name="phone_detector"
)

best_model_path = "/content/runs/detect/phone_detector/weights/best.pt"

print("Best model:", best_model_path)

## 11. Model Validation Results

The completed training run achieved:

| Metric | Score |
|---|---:|
| Precision | 86.7% |
| Recall | 81.2% |
| mAP@50 | 85.5% |
| mAP@50-95 | 60.4% |

These are the validation results from the completed 10-epoch training run.

## 12. Test the Model on a Real Image

Upload a phone image and run inference with the trained model.

In [ ]:
uploaded = files.upload()

image_path = next(
    (f"/content/{name}" for name in uploaded
     if name.lower().endswith((".jpg", ".jpeg", ".png", ".jfif"))),
    None
)

if image_path is None:
    raise FileNotFoundError("Please upload an image.")

# Convert JFIF to JPG when necessary.
if image_path.lower().endswith(".jfif"):
    from PIL import Image
    converted_path = "/content/phone_test.jpg"
    Image.open(image_path).save(converted_path)
    image_path = converted_path

model = YOLO(best_model_path)

prediction_results = model.predict(
    source=image_path,
    conf=0.5,
    save=True
)

result_image = prediction_results[0].plot()

plt.figure(figsize=(8, 6))
plt.imshow(result_image)
plt.axis("off")
plt.title("Real-World Phone Detection")
plt.show()

## 13. Upload a Video for Tracking

The video will be processed using YOLOv8 detection and ByteTrack object tracking.

In [ ]:
uploaded = files.upload()

video_path = next(
    (f"/content/{name}" for name in uploaded
     if name.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))),
    None
)

if video_path is None:
    raise FileNotFoundError("Please upload a video.")

print("Video:", video_path)

## 14. Get Video Information

In [ ]:
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

cap.release()

print("FPS:", fps)
print("Total frames:", total_frames)
print("Video duration:", round(total_frames / fps, 2), "seconds")

## 15. Phone Tracking with ByteTrack

**Detection** answers: “Is there a phone in this frame?”

**Tracking** answers: “Is this the same phone as in previous frames?”

ByteTrack assigns a tracking ID to detected phones.

In [ ]:
model = YOLO(best_model_path)

tracking_results = model.track(
    source=video_path,
    tracker="bytetrack.yaml",
    conf=0.5,
    imgsz=640,
    stream=True
)

phone_ids_seen = set()

for result in tracking_results:
    if result.boxes.id is not None:
        track_ids = result.boxes.id.int().cpu().tolist()
        phone_ids_seen.update(track_ids)

print("Tracking completed.")
print("Phone IDs detected:", sorted(phone_ids_seen))

## 16. Duration Monitoring and CSV Event Logging

Video time is calculated from the frame number and FPS rather than computer wall-clock time.

A small tracking gap of up to 5 frames is allowed, and detections shorter than 0.5 seconds are ignored.

In [ ]:
model = YOLO(best_model_path)

tracking_results = model.track(
    source=video_path,
    tracker="bytetrack.yaml",
    conf=0.5,
    imgsz=640,
    stream=True
)

phone_data = {}
events = []

MAX_MISSING_FRAMES = 5
MIN_DURATION = 0.5

frame_number = 0

for result in tracking_results:
    frame_number += 1
    current_ids = set()

    if result.boxes.id is not None:
        track_ids = result.boxes.id.int().cpu().tolist()
        current_ids.update(track_ids)

        for phone_id in track_ids:
            if phone_id not in phone_data:
                phone_data[phone_id] = {
                    "start_frame": frame_number,
                    "last_seen_frame": frame_number
                }
            else:
                phone_data[phone_id]["last_seen_frame"] = frame_number

    for phone_id in list(phone_data):
        if phone_id not in current_ids:
            missing_frames = (
                frame_number - phone_data[phone_id]["last_seen_frame"]
            )

            if missing_frames > MAX_MISSING_FRAMES:
                start_frame = phone_data[phone_id]["start_frame"]
                end_frame = phone_data[phone_id]["last_seen_frame"]

                start_time = (start_frame - 1) / fps
                end_time = end_frame / fps
                duration = end_time - start_time

                if duration >= MIN_DURATION:
                    events.append({
                        "Phone ID": phone_id,
                        "Start Time (sec)": round(start_time, 2),
                        "End Time (sec)": round(end_time, 2),
                        "Duration (sec)": round(duration, 2)
                    })

                del phone_data[phone_id]

# Finish phones still visible at the end of the video.
for phone_id in list(phone_data):
    start_frame = phone_data[phone_id]["start_frame"]
    end_frame = phone_data[phone_id]["last_seen_frame"]

    start_time = (start_frame - 1) / fps
    end_time = end_frame / fps
    duration = end_time - start_time

    if duration >= MIN_DURATION:
        events.append({
            "Phone ID": phone_id,
            "Start Time (sec)": round(start_time, 2),
            "End Time (sec)": round(end_time, 2),
            "Duration (sec)": round(duration, 2)
        })

df = pd.DataFrame(events)

print("PHONE MONITORING REPORT")
display(df)

csv_path = "/content/phone_monitoring_log.csv"
df.to_csv(csv_path, index=False)

print("CSV saved:", csv_path)

## 17. Generate Final Monitoring Video with Alerts

A warning is displayed when a phone remains visible for **3 seconds or longer**.

The final output is resized to 1280×720 to make processing and playback easier.

In [ ]:
model = YOLO(best_model_path)

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)

cap.release()

output_folder = "/content/final_phone_monitoring"
os.makedirs(output_folder, exist_ok=True)

output_path = os.path.join(
    output_folder,
    "phone_monitoring_final.mp4"
)

output_width = 1280
output_height = 720

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (output_width, output_height)
)

tracking_results = model.track(
    source=video_path,
    tracker="bytetrack.yaml",
    conf=0.5,
    imgsz=640,
    stream=True
)

phone_start_frames = {}
ALERT_THRESHOLD = 3.0
frame_number = 0

for result in tracking_results:
    frame_number += 1

    current_time = (frame_number - 1) / fps
    frame = result.plot()

    phone_count = 0

    if result.boxes.id is not None:
        track_ids = result.boxes.id.int().cpu().tolist()
        boxes = result.boxes.xyxy.cpu().tolist()

        phone_count = len(track_ids)

        for phone_id, box in zip(track_ids, boxes):

            if phone_id not in phone_start_frames:
                phone_start_frames[phone_id] = frame_number

            start_frame = phone_start_frames[phone_id]

            duration = (
                frame_number - start_frame + 1
            ) / fps

            x1, y1, x2, y2 = map(int, box)

            if duration >= ALERT_THRESHOLD:
                status = "WARNING: PHONE DETECTED TOO LONG"
            else:
                status = "Phone detected"

            cv2.putText(
                frame,
                f"ID: {phone_id} | Duration: {duration:.2f}s",
                (x1, max(y1 - 35, 30)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

            cv2.putText(
                frame,
                status,
                (x1, max(y1 - 10, 55)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 0, 255) if duration >= ALERT_THRESHOLD
                else (255, 255, 255),
                2
            )

    frame = cv2.resize(
        frame,
        (output_width, output_height)
    )

    cv2.putText(
        frame,
        f"Phones Detected: {phone_count}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Video Time: {current_time:.2f}s",
        (20, 75),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    out.write(frame)

out.release()

print("Final monitoring video created!")
print(output_path)

## 18. Download the Results

The following files can be downloaded from Colab:

- Final monitoring video
- CSV monitoring log
- Trained YOLO model

In [ ]:
print("Final video:", output_path)
print("CSV log:", csv_path)
print("Best model:", best_model_path)

# Uncomment the file you want to download:

# files.download(output_path)
# files.download(csv_path)
# files.download(best_model_path)

## 19. Conclusion

This project demonstrates a complete Computer Vision pipeline:

**Dataset Preparation → Annotation Cleaning → YOLOv8 Training → Real-World Detection → ByteTrack Tracking → Duration Monitoring → Alerts → CSV Logging → Streamlit Application**

### Technologies

Python • OpenCV • YOLOv8 • Ultralytics • ByteTrack • Pandas • Google Colab • Streamlit

The trained model achieved **86.7% precision, 81.2% recall, 85.5% mAP@50, and 60.4% mAP@50-95** on the validation set.